In [2]:
import torch
from torch import nn 
from torchvision import models
from torch.utils.data import DataLoader, Dataset
import os
import cv2
import h5py 
import numpy as np 
import matplotlib.pyplot as plt
from preprocess import ecg_processing_pipeline, smart_pad_and_resize_ecg
import pandas as pd
from albumentations.core.transforms_interface import ImageOnlyTransform
import random
import warnings
import sys
import albumentations as A 
from transformations import CornerCutout, GradientShadow, PaperFoldEffect, BottomBlur
from albumentations.pytorch import ToTensorV2

### Helpers

In [3]:
def check_device():
    """
    Check available compute devices and return the best one.
    Priority: CUDA > MPS > CPU
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("✓ CUDA available")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("✓ MPS (Apple Silicon GPU) available")
    else:
        device = torch.device("cpu")
        print("✗ Using CPU (no GPU acceleration available)")
    
    print(f"\nSelected device: {device}")
    return device

# Check and get device
device = check_device()

✓ MPS (Apple Silicon GPU) available

Selected device: mps


### Architecture

In [4]:
class Head(nn.Module):
    def __init__(self, in_features, hidden_layer, dropout_rate=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden_layer),
            nn.BatchNorm1d(hidden_layer),  
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer, hidden_layer // 2), 
            nn.BatchNorm1d(hidden_layer // 2),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer // 2, 1)
        )
    
    def forward(self, x):
        return self.layers(x)

class MultiHeadEfficientNet(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3):
        super().__init__()
        
        backbone = models.efficientnet_v2_s(weights="DEFAULT")
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        
        self.backbone = backbone
        
        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features), 
            nn.BatchNorm1d(in_features),
            nn.GELU(), 
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate)
        )
        
        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate) 
            for _ in range(num_conditions)
        ])
        
    def forward(self, x):
        backbone_feats = self.backbone(x)
        processed_feats = self.shared_feature_processor(backbone_feats)
        outputs = [head(processed_feats) for head in self.heads]
        return torch.cat(outputs, dim=1)  # [batch, num_conditions]

### Image Preprocessing/DataLoader

In [5]:
def load_contours_from_hdf5(filepath='contours.h5'):
    """
    Load all contours from HDF5 file back into dictionary format
    """
    contour_dict = {}
    
    with h5py.File(filepath, 'r') as f:
        for img_id in f.keys():
            grp = f[img_id]
            
            contour_dict[img_id] = {
                'contour': grp['contour'][:],  # Load the contour array
                'scale_x': grp.attrs['scale_x'],
                'scale_y': grp.attrs['scale_y'], 
                'half': grp.attrs['half']
            }
    
    return contour_dict

# Usage
contours = load_contours_from_hdf5(os.path.join(os.path.abspath("../"), "seg_model", 'image_preprocessing', 'contours.h5'))

In [6]:
def process_single_img(
                img,
                img_id,
                desired_aspect=0.5,
                target_width=512
                ):
    value = f"train_{str(img_id).zfill(6)}.png"
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    output = ecg_processing_pipeline(input_image = img, 
                                contour_data = contours[value.split(".")[0]])
    resize_output = smart_pad_and_resize_ecg(output, target_size=(int(target_width*desired_aspect), 512), resize_strategy=cv2.INTER_AREA)
    return resize_output

class ECGDataset(Dataset): 
    def __init__(self,
                image_paths, 
                labels_df, 
                transforms=None):
        
        self.image_paths = image_paths
        self.labels_dict = {idx: torch.tensor(row.values, dtype=torch.float32) 
                            for idx, row in labels_df.iterrows()}
        
        self.idx_to_image_id = {}
        for idx, path in enumerate(self.image_paths):
            filename = os.path.basename(path)
            image_id = int(filename.rsplit('_', 1)[-1].split('.')[0])
            self.idx_to_image_id[idx] = image_id
        
        self.transforms = transforms
        
    def __len__(self): 
        return len(self.image_paths)
        
    def __getitem__(self, idx): 
        image_path = self.image_paths[idx]
        index = self.idx_to_image_id[idx]
        label = self.labels_dict[index]
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")
        # convert image using our segmentation pipeline 
        image_conv = process_single_img(
            img=image, 
            img_id=index,
            desired_aspect=0.5, 
            target_width=512
        )
        image_conv = np.stack([image_conv, image_conv, image_conv], axis=-1)
        if self.transforms is not None: 
            image_tens = self.transforms(image=image_conv)["image"]
        return image_tens, label

In [7]:
train_transforms = A.Compose([
    # Data Augmentations
    A.Rotate(limit=5, p=0.3),  # small rotations, limit is +/- degrees
    A.Affine(translate_percent={'x': (-0.1, 0.1), 'y': (-0.1, 0.1)}, 
            rotate=0, scale=1.0, shear=0, p=0.3),  # small translations
    A.Perspective(scale=(0.05, 0.1), p=0.4), # perspective transformations
    CornerCutout(max_cut_size=0.15, num_corners=2, p=0.2),  # Add corner cutout
    GradientShadow(intensity=0.3, p=0.2), # lighting gradient over the page
    A.RandomShadow( # shadows
        shadow_roi=(0, 0, 1, 1),  # Can appear anywhere in image
        num_shadows_limit=(1, 2),  # 1-2 shadow regions
        shadow_dimension=5,         # Controls shadow size/complexity
        p=0.2
    ),
    # blur at bottom of page
    BottomBlur(bottom_region=0.3, p=0.3),
    # "paper fold" effect
    PaperFoldEffect(p=0.2),
    # Normalization (ImageNet)
    A.Normalize(mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225]),
    # Convert to tensor and replicate grayscale to 3 channels
    ToTensorV2()
])

val_transforms = A.Compose([
    # Normalization (ImageNet)
    A.Normalize(mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225]),
    # Convert to tensor
    ToTensorV2()
])

In [8]:
data_path = os.path.abspath(os.path.join("../", "data", "train"))
ids = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
paths = [os.path.join(data_path, f"train_{str(id_val).zfill(6)}.png") for id_val in ids]
label_df = pd.read_csv(os.path.join("../", "data", "train_final.csv"), index_col=0)
train_dataset = ECGDataset(image_paths=paths, 
                    labels_df=label_df, 
                    transforms=train_transforms, 
                    )
val_dataset = ECGDataset(image_paths=paths, 
                    labels_df=label_df, 
                    transforms=val_transforms, 
                    ) # obvs need to be changed!!
warnings.warn("Using the wrong datasets, just to get the code running: make sure you change it", UserWarning)

/var/folders/7s/40ntfhvj5c32kvn0xr_d_wxm0000gn/T/ipykernel_22698/1683026307.py:13: UserWarning: Using the wrong datasets, just to get the code running: make sure you change it
  warnings.warn("Using the wrong datasets, just to get the code running: make sure you change it", UserWarning)


In [9]:
num_workers = 0 if sys.platform == 'darwin' else 4 
train_dataloader = DataLoader( 
                        train_dataset,
                        batch_size=16, 
                        shuffle=True, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)
val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=16, 
                        shuffle=True, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

### Testing

In [ ]:
from losses import AsymmetricLossOptimized

criterion = AsymmetricLossOptimized(
    gamma_neg=4, 
    gamma_pos=1,
    clip=0.05)
criterion

SyntaxError: invalid syntax. Perhaps you forgot a comma? (971246393.py, line 5)